<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/02-rdd-fundamentals/01-transformations-actions-lazy.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 2.1 — Transformations vs actions, lazy evaluation, DAG

Run `spark_ui(spark)` once the session exists and keep the Spark UI open in a second tab for this one —
the whole point is watching when jobs appear.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("2.1-lazy-evaluation")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

LOGS = f"{DATA_DIR}/sample_logs.txt"

## Transformations return instantly — they only record a plan

In [ ]:
import time

big = sc.parallelize(range(100_000_000))   # 100M elements

t0 = time.perf_counter()
transformed = big.map(lambda x: x * 2).filter(lambda x: x % 6 == 0)
print(f"two transformations on 100M elements: {time.perf_counter() - t0:.6f}s")
# microseconds — nothing ran. Check the Jobs tab: no new job.

In [ ]:
t0 = time.perf_counter()
n = transformed.count()                    # ACTION
print(f"count() = {n:,} in {time.perf_counter() - t0:.2f}s")
# THAT took time — and the Jobs tab now shows the job.

## Lineage: the DAG each RDD carries

In [ ]:
lines  = sc.textFile(LOGS)
warns  = lines.filter(lambda l: " WARN " in l)
pairs  = warns.map(lambda l: (l.split()[3], 1))
counts = pairs.reduceByKey(lambda a, b: a + b)

print(counts.toDebugString().decode())   # read bottom-up: file -> filter/map -> shuffle

## Symptom 1: errors surface at the action

The bad path below is accepted silently; the failure only fires in `count()`.
Note the exception names the real problem (the path) even though the
traceback points at the action line.

In [ ]:
ghost = sc.textFile("definitely/not/a/real/path")
cleaned = ghost.map(str.strip).filter(lambda l: l)   # still no error!

try:
    cleaned.count()
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

## Symptom 2 & 3: every action re-runs the whole lineage

Note the Jobs-tab count before running, then after: **two** new jobs,
each executing the full read-filter-map-shuffle pipeline.

In [ ]:
print(counts.collect())   # job 1: full pipeline
print(counts.collect())   # job 2: the SAME full pipeline, again
# RDDs remember recipes, not results. Fix arrives in lesson 2.4 (cache).

In the UI, open the job you just ran: it has **two stages** — the
`reduceByKey` shuffle is the boundary (lesson 0.5). One action → one job →
stages at shuffle boundaries → one task per partition.

## Exercises

1. Predict the number of NEW jobs this creates, then verify in the UI:
   `x = lines.map(str.upper)`, `x.take(2)`, `y = x.filter(lambda l: 'API' in l)`, `y.count()`, `print(y.first())`.
2. Use `toDebugString()` on `y` from exercise 1. How many distinct RDDs are in its lineage?
3. Write a pipeline over the log file that extracts `latency_ms` values (hint: only some lines have them) and computes the max — using exactly ONE action.

In [ ]:
# your exercise code here
